In [1]:
# Required Python packages
from datetime import date, datetime, timedelta
import calendar
import pandas as pd
import numpy as np
import itertools
import copy
from numpy.linalg import multi_dot
from scipy.stats import norm
from scipy.stats import bernoulli
from scipy.optimize import fmin_slsqp as min

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm
from statsmodels.api import OLS

from sklearn.linear_model import LassoCV
from yellowbrick.datasets import load_concrete
from yellowbrick.regressor import AlphaSelection

import matplotlib.pyplot as plt
import math


#Import Date opertative user defined functions
from ipynb.fs.full.user_defined_vik_functions import get_all_monthly_option_expiries, \
                                                     find_last_thurs_date_of_month, \
                                                     prev_workday_if_holiday, find_wkly_expries,\
                                                     date_of_prev_thurs

# Import dataframe naming functions 
from ipynb.fs.full.user_defined_vik_functions import get_mthly_df_name_from_expiry

#Import data loading functions
from ipynb.fs.full.user_defined_vik_functions import load_all_mthly_data

#import traded options parameters and info
from ipynb.fs.full.user_defined_vik_functions import generate_weekly_strikes, generate_mthly_strikes

#Import pricing functions
from ipynb.fs.full.user_defined_vik_functions import BSM_call_vec_with_div, BSM_put_vec_with_div

#Import implied volatility processing functions
from ipynb.fs.full.user_defined_vik_functions import generate_weekly_iv, gen_bivariate_forward_smile

#Import Risk-free interest rate generation functions
from ipynb.fs.full.user_defined_vik_functions import generate_weekly_ir

#Import Stochatic process simulation functions
from ipynb.fs.full.user_defined_vik_functions import gbm1W_simulation, generate_covariance_from_correlation

# Import risk free interest rate function
from ipynb.fs.full.user_defined_vik_functions import get_risk_free_rate_from_exact_date


In [2]:
# Stock Index of Interest
stock_ident = "BANKNIFTY"

#Static hedging performed at different moneyness regions 
#i.e. moneyness is used to select the option with nearest moneyness match
# ATM - At the Money, ITM - In the money, OTM - Out of the Money
prod_moneyness = "OTM"

#Product type to hedge: either "CE" or "PE"
prod_type = "PE"
# prod_type = "PE"

# The scope of this code is to hedge one option 
# and scope will be extended to a portfolio
no_of_assets = 1
cor_mat = [[1]]

#Path to refer data
source_path = "/home/jupyter-partha/Vikranth - Chapter 2/"
input_sub_path = "Input Data/mkt_data_covid_region/"
output_sub_path = "Output Data/"
input_data_path = source_path + input_sub_path
output_data_path = source_path + output_sub_path

# Periods of interest will be a dictionary
#Key is the year, value is a list of months 1-12, 1- Jan, 2 - Feb,...12 - Dec
# For E.g., periods_of_interest = {2020: [3], 2019: [11, 12]}
periods_of_interest = {2019: [8, 9, 10, 11, 12], 2020: [1, 2, 3, 4, 5, 6, 7]}

#List of holidays
holidays_list = [date(2019, 3, 4), date(2019, 3, 21),\
                 date(2019, 4, 17), date(2019, 4, 19), date(2019, 4, 29),\
                 date(2019, 5, 1),\
                 date(2019, 6, 5),\
                 date(2019, 8, 12), date(2019, 8, 15),\
                 date(2019, 9, 2), date(2019, 9, 10), \
                 date(2019, 10, 2), date(2019, 10, 8), date(2019, 10, 21), date(2019, 10, 28), \
                 date(2019, 11, 12), \
                 date(2019, 12, 25), \
                 date(2020, 2, 21), \
                 date(2020, 3, 10), \
                 date(2020, 4, 2), date(2020, 4, 6), date(2020, 4, 10), date(2020,4, 14), \
                 date(2020, 5, 1), date(2020, 5, 25), \
                 date(2020, 10, 2), date(2020, 11, 16), date(2020, 11, 30), date(2020, 12, 25)]
#Simulation Parameters
no_of_paths = 5000

no_opt=1


In [3]:
#########################
#User-defined functions #
#########################

def generate_weekly_static_pfl_weights(no_of_paths, mthly_expiries_list, \
                                       dict_wkly_expiries_each_month, \
                                       dict_ce_wkly_strikes, dict_pe_wkly_strikes, \
                                       dict_wkly_spots, dict_wkly_iv, \
                                       prod_type, prod_moneyness, output_path, \
                                       option_type = "CE", no_of_assets=1, cor_mat=[[1]],\
                                       stock_ident="BANKNIFTY"):
    ce_pfl_weights = []
    pe_pfl_weights = [] 
    cash_weights = []
    for each_month in mthly_expiries_list:
        strike = dict_mthly_strikes[each_month.strftime("%d-%b-%Y")]
        weekly_pfl_weights = []
        ce_dict = {}
        pe_dict = {}
        cash_dict = {}
        week_list = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")]
        no_of_weeks = len(week_list)
        
        for week in range(0, no_of_weeks):
            s_t = dict_wkly_spots[each_month.strftime("%d-%b-%Y")][week]
            K = strike
            vol_list = np.array([dict_wkly_iv[each_month.strftime("%d-%b-%Y")][week]])
            r = np.array([dict_wkly_ir[each_month.strftime("%d-%b-%Y")][week]])
            
            if (week == no_of_weeks-1):
                sim_start_date = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")][week]
                sim_end_date = each_month
            else:
                sim_start_date = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")][week]
                sim_end_date = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")][week + 1]
         
            sim_stock_mat = gbm1W_simulation(no_of_paths, sim_start_date, sim_end_date, \
                                             no_of_assets, s_t, r, vol_list, cor_mat)
            
            call_strikes = np.sort(np.array(dict_ce_wkly_strikes[each_month.strftime("%d-%b-%Y")][week])).reshape(1,-1)
            put_strikes = np.sort(np.array(dict_pe_wkly_strikes[each_month.strftime("%d-%b-%Y")][week])).reshape(1,-1)
            no_of_calls = call_strikes.size
            no_of_puts = put_strikes.size
            no_ind_vars = no_of_calls + no_of_puts
            
            
            stock_vec = sim_stock_mat[:, 1]
            r_f = get_risk_free_rate_from_exact_date(sim_start_date)
            dt = float((each_month - sim_end_date).days) / 365
            opt_strike = strike
    
            df_daily_iv_data = pd.read_csv(output_path + "A1_" + stock_ident + "_" +  "Daily_vol_smile_" + option_type + "_" + each_month.strftime("%d-%b-%Y") + ".csv")
            df_daily_iv_data = df_daily_iv_data[df_daily_iv_data['Impl_Vol'] > 0.00001]

            
            spot_list = sim_stock_mat[:,1]
            if (prod_type == "CE"):
                mny_list = spot_list / K
            else:
                mny_list = np.reciprocal(spot_list) * K

            if (week != no_of_weeks-1):
                date_str = sim_start_date.strftime("%d-%b-%Y")
                gen_iv = lambda x: gen_bivariate_forward_smile(df_daily_iv_data, date_str, x, dt)
                gen_iv = np.vectorize(gen_iv)
                vol_arr = gen_iv(mny_list)
            else:
                vol_arr = np.array([0]) 
                
            if (option_type == "CE" and week != no_of_weeks-1):
                option_value = BSM_call_vec_with_div(stock_vec, opt_strike, r, r_f, vol_arr, dt)
            elif (option_type == "PE" and week != no_of_weeks-1):
                option_value = BSM_put_vec_with_div(stock_vec, opt_strike, r, r_f, vol_arr, dt) 
            elif (option_type == "CE" and week == no_of_weeks-1):
                option_value = np.maximum(stock_vec - opt_strike, 0) 
            else:
                option_value = np.maximum(opt_strike - stock_vec, 0)
    
            option_value = np.asarray(option_value)
            option_value = option_value.reshape(-1, 1) 
            
            y = option_value
            stock_vec = stock_vec.reshape(-1,1)
            x = np.concatenate((np.maximum(stock_vec-call_strikes,0), np.maximum(put_strikes-stock_vec,0)), axis=1) 

            print("SKLearn Linear regression - Lasso")
            lin_model_lasso = Lasso(alpha=1.0).fit(x, y)
            pfl_weights = lin_model_lasso.coef_
            pfl_weights = pfl_weights.reshape(-1)
            cash_weight = lin_model_lasso.intercept_
            print("R-Square: ", lin_model_lasso.score(x, y))
            print("Coefficients:")
            print(pfl_weights)
            print(cash_weight)

#             #Find Optimal Alpha
#             alphas = np.array([-1.0, -0.5, 0.0, 0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0])
#             model = LassoCV(alphas=alphas)
#             visualizer = AlphaSelection(model)
#             visualizer.fit(x, y)
#             visualizer.show()    

            # Call Pfl weights
            call_strikes = call_strikes.reshape(-1) 
            ce_dates_list = np.array([sim_start_date.strftime("%d-%b-%Y") for i in range(0, no_of_calls)]).reshape(-1)
            ce_wt_list = pfl_weights[0:no_of_calls]
            ce_dict = {"Date": ce_dates_list, "Strike": call_strikes, "Pfl_weights": ce_wt_list}
            df_ce = pd.DataFrame(ce_dict)
            ce_pfl_weights.append(df_ce)
    
            
            #Put Pfl Weights
            put_strikes = put_strikes.reshape(-1) 
            pe_dates_list = np.array([sim_start_date.strftime("%d-%b-%Y") for i in range(0, no_of_puts)]).reshape(-1)
            pe_wt_list = pfl_weights[no_of_calls:no_ind_vars]
            pe_dict = {"Date": pe_dates_list, "Strike":put_strikes, "Pfl_weights": pe_wt_list}
            df_pe = pd.DataFrame(pe_dict)
            pe_pfl_weights.append(df_pe)
            
            # Cash component
            cash_date = [sim_start_date.strftime("%d-%b-%Y")]
            cash_dict = {"Date":cash_date, "Pfl_weights":cash_weight} 
            df_cash = pd.DataFrame(cash_dict)
            cash_weights.append(df_cash)
            
            
        df_ce_pfl_weights = pd.concat(ce_pfl_weights, axis=0)
        df_pe_pfl_weights = pd.concat(pe_pfl_weights, axis=0)
        df_cash_pfl_weights = pd.concat(cash_weights, axis=0)

        df_ce_pfl_weights.to_csv(output_path  + "B5_LR_" + stock_ident + "_" + prod_moneyness + "_" + prod_type + "_CE_Pfl_weights_Fwd_Smile.csv", index = False)
        df_pe_pfl_weights.to_csv(output_path + "B5_LR_" + stock_ident + "_" + prod_moneyness + "_" + prod_type + "_PE_Pfl_weights_Fwd_Smile.csv", index = False)
        df_cash_pfl_weights.to_csv(output_path  + "B5_LR_" + stock_ident + "_" + prod_moneyness + "_" + prod_type + "_Cash_Pfl_weights_Fwd_Smile.csv", index = False)
    
    return df_ce_pfl_weights, df_pe_pfl_weights, df_cash_pfl_weights

def output_strikes_hedged(prod_type, prod_moneyness, dict_mthly_strikes, stock_ident, output_path):
    list_of_dates = []
    list_of_prod_type = []
    list_of_strikes = []
    for each_month in dict_mthly_strikes.keys():
        list_of_dates.append(each_month)
        list_of_strikes.append(dict_mthly_strikes[each_month])
        list_of_prod_type.append(prod_type)
    dict_hedged_prod = {"Date":list_of_dates, "prod_type":list_of_prod_type, "Strike":list_of_strikes}
    df_hedged_prod = pd.DataFrame(dict_hedged_prod) 
    df_hedged_prod.to_csv(output_path + "B5_LR_" + stock_ident + "_" + prod_moneyness + "_" + prod_type + "_strikes_hedged.csv", index = False)
    return(df_hedged_prod)


In [4]:

#Find the monthly strikes of option from mkt data to find the option to be hedged
#Every month, an option is hedged
mthly_expiries_list = get_all_monthly_option_expiries(periods_of_interest, holidays_list)
dict_wkly_expiries_each_month = find_wkly_expries(mthly_expiries_list, holidays_list)

#Load all monthly mkt data
mthly_mkt_data = load_all_mthly_data(mthly_expiries_list, input_data_path, holidays_list, prod_type_lists=["FUT", "CE", "PE"], stock_ident=stock_ident)

#Load weekly strikes of short term options used for hedging monthly options 
dict_ce_wkly_strikes, dict_pe_wkly_strikes, dict_wkly_spots = generate_weekly_strikes(dict_wkly_expiries_each_month, input_data_path, stock_ident = "BANKNIFTY")     

#Load monthly strikes of interest as of required moneyness on the begining of month
dict_mthly_strikes = generate_mthly_strikes(mthly_mkt_data, prod_moneyness, prod_type, holidays_list, stock_ident=stock_ident)

#Load weekly IV for Black-Scholes model input and monte-carlo simulation of stock paths
dict_wkly_iv = generate_weekly_iv(dict_wkly_expiries_each_month, dict_wkly_spots, dict_mthly_strikes, prod_type, output_path=output_data_path, atm_ind=0, stock_ident=stock_ident)

#Load weekly risk free interest rates from futures
#This function to be changed based on updates on risk-free-interest - now kept it as cosntant
dict_wkly_ir = generate_weekly_ir(dict_wkly_expiries_each_month, mthly_mkt_data, stock_ident)

#Load Static portfolio weights built by neural network
ce_pfl_wts, pe_pfl_wts, cash_pfl_wts= generate_weekly_static_pfl_weights(no_of_paths, mthly_expiries_list, dict_wkly_expiries_each_month, \
                                                        dict_ce_wkly_strikes, dict_pe_wkly_strikes, dict_wkly_spots, \
                                                        dict_wkly_iv, prod_type, prod_moneyness, output_data_path, \
                                                        option_type = prod_type, no_of_assets=no_of_assets, \
                                                        cor_mat=cor_mat, stock_ident=stock_ident) 
# The strike correspnding to the option that is hedged
df_hedged_prod = output_strikes_hedged(prod_type, prod_moneyness, dict_mthly_strikes, stock_ident, output_data_path)


SKLearn Linear regression - Lasso


/opt/tljh/user/lib/python3.7/site-packages/sklearn/linear_model/_coordinate_descent.py:532: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 5184.750051789779, tolerance: 3160.9226685728217
  positive)


R-Square:  0.9993432153045069
Coefficients:
[-0.08485923  0.01110394 -0.00724059 -0.00260939  0.00437266  0.01374696
  0.00911239  0.0076371   0.00681613  0.00605883  0.0053727   0.00458814
  0.00399986  0.00350881  0.00293763  0.00245073  0.00284315  0.00298108
  0.          0.          0.          0.          0.          0.
  0.          0.         -0.10434252  0.09342494  0.04664167  0.02220473
  0.04104192  0.          0.03442802  0.06801307  0.          0.02088573
  0.04053996 -0.02483805  0.01539828  0.00477182  0.03287634  0.11211369
  0.01899737 -0.01632222]
[83.42780717]
SKLearn Linear regression - Lasso


/opt/tljh/user/lib/python3.7/site-packages/sklearn/linear_model/_coordinate_descent.py:532: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 245605.9928349937, tolerance: 17359.442249596792
  positive)


R-Square:  0.996437749866343
Coefficients:
[-0.21187019  0.01531782  0.02289322  0.11469444 -0.09294027 -0.0031641
  0.00854481  0.02122609  0.01435811  0.01241665  0.01159348  0.01055941
  0.00954793  0.00860413  0.00760497  0.00662709  0.00617878  0.00537193
  0.00432701  0.0069653   0.0043318   0.          0.          0.
  0.42579598 -0.14331339  0.02207467  0.01790735  0.08244874 -0.14200782
  0.22249413  0.07551309 -0.09864286  0.000989    0.07418067  0.19303821
  0.01172589  0.0049757 ]
[205.61146006]
SKLearn Linear regression - Lasso
R-Square:  0.9998941149611578
Coefficients:
[-2.44870500e-01  1.95076869e-02  1.45109796e-02  3.83436855e-02
  3.33262241e-02  1.41956975e-02  4.15054540e-02  2.33990802e-04
 -2.71298134e-02 -0.00000000e+00  5.13053548e-02  6.52476980e-03
  6.97649059e-03  6.39074497e-03  4.30895624e-03  5.06729312e-03
  1.99475958e-03  0.00000000e+00  0.00000000e+00  4.93478399e-03
  0.00000000e+00  0.00000000e+00 -0.00000000e+00 -0.00000000e+00
 -0.00000000e+00  1

...

R-Square:  0.9979118671965623
Coefficients:
[-0.16956734 -0.04778316 -0.04219724  0.16098975  0.10059251 -0.01799093
 -0.06930272 -0.03429606  0.02677411  0.04591761  0.04044512  0.00364106
 -0.00400296 -0.00709849 -0.00123844 -0.00107156 -0.02210872 -0.00311007
  0.00300558  0.00950693  0.00386461  0.00300044  0.00475363  0.00558765
  0.0022213   0.0013251   0.0030974   0.          0.03428323  0.02715995
 -0.          0.23772666  0.11802894  0.07339997  0.04130722  0.01886462
  0.02687713  0.01886327  0.00523546  0.00737337  0.0251431   0.02651773
  0.03491414  0.02218161  0.00073449 -0.01448522 -0.0064078   0.03250017
  0.07335624  0.08731967  0.06863897  0.01806446 -0.05911617]
[174.25988177]
SKLearn Linear regression - Lasso


/opt/tljh/user/lib/python3.7/site-packages/sklearn/linear_model/_coordinate_descent.py:532: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 50520.04254372558, tolerance: 50453.007443483104
  positive)


R-Square:  0.9997999443927023
Coefficients:
[-1.01937166e-01  1.44316325e-02  1.50865274e-02  1.41430937e-02
  1.03277193e-02  8.40087674e-03  1.46787629e-02  1.13126588e-02
 -1.10941521e-03 -4.63964944e-03 -1.82652758e-03 -7.50042508e-04
  4.57816429e-04  4.88007938e-03  5.21372142e-03  2.89375632e-03
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  2.43919741e-04  8.18286829e-04  9.36856338e-04  1.02226811e-03
  9.18300221e-04  5.85010704e-04  3.64450876e-04  2.59651506e-04
  4.50174264e-04  2.82534706e-04  5.37526805e-04  6.67456327e-05
  1.88884131e-05  0.00000000e+00  0.00000000e+00  9.84514256e-04
  0.00000000e+00  1.09122674e-02 -6.12523662e-03  2.11406648e-01
  1.77615714e-01 -1.29209195e-01  5.58292086e-02  1.45960853e-01
 -1.98194677e-03  4.47559080e-04  7.31108076e-02  2.26545121e-02
  4.01643857e-02  3.12982966e-02  6.62860017e-03  2.43203255e-02
  1.35822932e-02  8.96978248e-03  3.55453530e-02  3.11467617e-02
  2.68104800e-02  1.99317569e-02  1.11691238e-

/opt/tljh/user/lib/python3.7/site-packages/sklearn/linear_model/_coordinate_descent.py:532: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 578298.1145082151, tolerance: 119571.4804321056
  positive)


R-Square:  0.9990128558066939
Coefficients:
[-3.60618531e-01  4.07023283e-02  3.49164822e-02  3.22463738e-02
  1.36049687e-02  3.95496373e-03  5.59573313e-03  7.56812565e-03
  8.67558666e-03  9.26805288e-03  9.49051390e-03  9.40407877e-03
  8.78438875e-03  8.13813172e-03  7.65958808e-03  7.28490415e-03
  6.63544447e-03  6.14761114e-03  6.11551527e-03  5.83750473e-03
  5.75730119e-03  6.08498323e-03  6.92501834e-03  7.23831567e-03
  6.05244068e-03  5.05530586e-03  4.70692008e-03  4.07329072e-03
  3.10448057e-03  2.20193049e-03  2.92866980e-03  5.21583098e-03
  7.84477300e-03  5.66970220e-04  0.00000000e+00  3.05993188e-02
  0.00000000e+00 -0.00000000e+00  2.65923236e-01 -8.04527669e-01
  7.27557723e-01  8.20692903e-02  6.42164893e-02 -4.29599113e-02
 -9.48250867e-02  2.53976772e-02  2.55161129e-01  1.59734236e-01
  3.34840291e-02 -8.96950846e-02 -9.47620075e-02 -6.27514477e-01
  9.89576894e-02  3.91531658e-01  7.83653878e-02  1.76073940e-02
 -1.91500850e-02  1.42745171e-02  7.10438001e-